# 10. LLM을 활용한 텍스트마이닝 기법


## 1. GPT API 이해하기

### API란?

API(Application Programming Interface)는 한 프로그램이 다른 프로그램의 기능을 사용할 수 있게 해주는 약속입니다. 웹사이트에서 날씨를 불러오거나, 결제 시스템과 연결하거나, 지도 정보를 가져오는 것도 API를 통해 이루어집니다.

OpenAI의 GPT API는 내 컴퓨터에 GPT 모델을 직접 설치하지 않고, 코드에서 OpenAI 서버로 요청을 보내 답변을 받는 방식입니다.

```text
내 노트북 코드 -> OpenAI API 요청 -> GPT 모델 처리 -> 응답 반환 -> 파이썬에서 결과 활용
```

ChatGPT 웹/앱과 GPT API는 사용 방식이 다릅니다.

| 구분 | ChatGPT 웹/앱 | GPT API |
|---|---|---|
| 사용 방식 | 사람이 화면에서 직접 대화 | 파이썬 코드가 모델을 호출 |
| 주요 목적 | 개인 업무 보조, 대화 | 서비스 개발, 자동화, 데이터 처리 |
| 과금 기준 | ChatGPT 구독 요금제 | API 사용량 기반 별도 과금 |
| 결과 활용 | 화면에서 읽고 복사 | 데이터프레임, 파일, 앱, 서비스에 연결 |

GPT API를 사용하면 모델의 답변을 파이썬 코드 안에서 데이터로 받아 요약, 분류, 키워드 추출 같은 작업에 연결할 수 있습니다.


### API Key와 `.env` 파일

GPT API를 사용하려면 API Key가 필요합니다. API Key는 OpenAI가 `누가 요청을 보냈는지`, `얼마나 사용했는지`, `어느 계정에 과금해야 하는지`를 확인하는 비밀 인증 값입니다.

API Key는 비밀번호처럼 다뤄야 합니다.

- 노트북 코드 안에 직접 적지 않습니다.
- GitHub, 블로그, 캡처 화면에 노출하지 않습니다.
- 다른 사람에게 공유하지 않습니다.
- 유출이 의심되면 즉시 삭제하고 새로 발급합니다.

실습 폴더 루트에 `.env` 파일을 만들고 아래처럼 저장합니다.

```bash
OPENAI_API_KEY=sk-...
```

아래 셀의 `load_dotenv()`는 `.env` 파일에 저장된 API Key를 현재 파이썬 실행 환경으로 불러옵니다.


### GPT API 모델과 요금 이해하기

OpenAI API에서는 `model` 값으로 어떤 모델을 사용할지 정합니다. 예를 들어 `ChatOpenAI(model="gpt-4o-mini")`라고 쓰면 LangChain이 OpenAI의 `gpt-4o-mini` 모델에 요청을 보냅니다.

모델을 고를 때는 아래 기준을 함께 봅니다.

- 성능: 복잡한 추론, 긴 글 이해, 코드 작성이 필요한가?
- 비용: 같은 작업을 많이 반복할 예정인가?
- 속도: 답변이 빠르게 돌아와야 하는가?
- 입력 형태: 텍스트만 쓰는가, 이미지도 넣는가?
- 출력 형태: 자유 문장인가, JSON 같은 정해진 형식인가?

아래 가격은 2026-05-27 기준 USD 가격입니다. 최신 가격은 공식 Pricing 문서에서 확인할 수 있습니다.

| 모델 | 설명 | 입력 가격 / 1M tokens | 출력 가격 / 1M tokens | 추천 상황 |
|---|---|---:|---:|---|
| `gpt-5.5` | 최신 플래그십 모델. 복잡한 추론과 코딩에 강하지만 비용이 높습니다. | $5.00 | $30.00 | 어려운 분석, 고난도 추론, 중요한 코드 작업 |
| `gpt-5.4` | 고성능 작업용 모델. `gpt-5.5`보다 저렴한 상위 모델입니다. | $2.50 | $15.00 | 품질이 중요한 문서 분석, 전문 업무 자동화 |
| `gpt-5.4-mini` | 성능과 비용의 균형을 맞춘 소형 모델입니다. | $0.75 | $4.50 | 대량 처리, 업무형 챗봇, 실무 자동화 |
| `gpt-5.4-nano` | 단순하고 반복적인 작업에 적합한 저비용 모델입니다. | $0.20 | $1.25 | 분류, 태깅, 정보 추출, 빠른 대량 처리 |
| `gpt-4o-mini` | 빠르고 저렴한 소형 모델입니다. | $0.15 | $0.60 | 입문 실습, 요약, 키워드 추출, 감정 분류 |
| `text-embedding-3-small` | 텍스트를 숫자 벡터로 바꾸는 임베딩 모델입니다. | $0.02 | 해당 없음 | 유사도 검색, 추천, RAG 검색 단계 |

### 토큰이란?

토큰은 모델이 글을 읽고 쓸 때 사용하는 작은 단위입니다. 한국어에서는 글자, 단어 조각, 띄어쓰기 등이 섞여 토큰으로 나뉩니다.

요금은 보통 다음처럼 계산됩니다.

```text
총 비용 = 입력 토큰 비용 + 출력 토큰 비용
입력 토큰 비용 = 입력 토큰 수 / 1,000,000 * 입력 단가
출력 토큰 비용 = 출력 토큰 수 / 1,000,000 * 출력 단가
```

예를 들어 `gpt-4o-mini`에 입력 1,000 tokens, 출력 300 tokens를 사용했다면 대략 다음 비용이 듭니다.

```text
입력 비용: 1,000 / 1,000,000 * $0.15 = $0.00015
출력 비용:   300 / 1,000,000 * $0.60 = $0.00018
총 비용: $0.00033
```

반복 실행하면 비용이 누적됩니다. 긴 본문을 넣을 때는 `article_limit`, `body_char_limit`처럼 입력 길이를 제한하는 변수를 활용할 수 있습니다.

공식 참고 링크:

- Models: https://developers.openai.com/api/docs/models
- Pricing: https://developers.openai.com/api/docs/pricing
- GPT-4o mini: https://developers.openai.com/api/docs/models/gpt-4o-mini
- Embeddings: https://developers.openai.com/api/docs/models/text-embedding-3-small


In [ ]:
# API를 호출하지 않는 간단 비용 계산 예시입니다.
# gpt-4o-mini 기준: 입력 $0.15 / 1M tokens, 출력 $0.60 / 1M tokens

input_tokens = 1000
output_tokens = 300
input_price_per_1m = 0.15
output_price_per_1m = 0.60

input_cost = input_tokens / 1_000_000 * input_price_per_1m
output_cost = output_tokens / 1_000_000 * output_price_per_1m
total_cost = input_cost + output_cost

print(f"입력 비용: ${input_cost:.6f}")
print(f"출력 비용: ${output_cost:.6f}")
print(f"총 예상 비용: ${total_cost:.6f}")
print(f"같은 요청을 1,000번 실행하면 약 ${total_cost * 1000:.3f}입니다.")


In [ ]:
from dotenv import load_dotenv  # .env 파일에 저장된 환경변수를 불러오는 함수입니다.

load_dotenv()  # 현재 폴더의 .env 파일을 읽어 OpenAI API 키를 사용할 수 있게 합니다.


### LangChain 기본 구성

LangChain에서는 보통 다음 세 가지를 연결해 사용합니다.

- `ChatPromptTemplate`: 모델에게 전달할 지시문과 입력 형식
- `ChatOpenAI`: OpenAI 채팅 모델 호출
- `StrOutputParser` 또는 `JsonOutputParser`: 응답을 문자열이나 JSON 형태로 정리

아래 코드는 이번 차시에서 반복해서 사용할 기본 객체를 준비합니다.


### 시스템 프롬프트란?

`ChatPromptTemplate.from_messages()`에서는 메시지를 역할별로 나눠 작성합니다.

- `system`: 모델의 기본 역할, 말투, 답변 기준을 정하는 지시문입니다.
- `user`: 실제 사용자가 요청하는 질문이나 분석 대상 텍스트입니다.
- `assistant`: 이전 답변 예시를 넣을 때 사용합니다. 이번 실습에서는 주로 `system`과 `user`만 사용합니다.

예를 들어 `("system", "너는 텍스트마이닝 분석가야.")`라고 쓰면, 이후 사용자 요청을 분석가 관점에서 처리하도록 방향을 잡아주는 역할을 합니다.  
수업에서는 시스템 프롬프트를 어렵게 생각하기보다 **모델에게 맡길 역할을 먼저 정하는 문장**으로 이해하면 됩니다.


In [ ]:
import json  # JSON 형태의 문자열과 파이썬 객체를 다룰 때 사용합니다.
import numpy as np  # 임베딩 벡터의 유사도 계산에 사용할 수치 연산 라이브러리입니다.
import pandas as pd  # 표 형태의 결과를 보기 좋게 정리하는 라이브러리입니다.

from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # OpenAI 채팅 모델과 임베딩 모델을 LangChain에서 사용합니다.
from langchain_core.prompts import ChatPromptTemplate  # system/user 메시지로 프롬프트를 구성합니다.
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser  # 모델 응답을 문자열 또는 JSON으로 정리합니다.

chat_model_name = "gpt-4o-mini"  # 실습 기본 모델입니다. 필요하면 "gpt-5.4-nano" 등으로 바꿔 비교할 수 있습니다.
llm = ChatOpenAI(model=chat_model_name, temperature=0)  # temperature=0은 답변을 비교적 일관되게 만듭니다.


## 2. 텍스트 요약

요약은 긴 리뷰, 뉴스, 회의록, 상담 기록에서 핵심 내용을 빠르게 파악할 때 자주 사용됩니다.  
요약 프롬프트에서는 **무엇을 남길지**, **얼마나 짧게 만들지**, **어떤 형식으로 출력할지**를 명확히 정합니다.


In [ ]:
sample_text = """
최근 한 달간 온라인 쇼핑몰의 생활가전 카테고리 리뷰 1,200건을 살펴보았다.
고객들은 대체로 제품 성능과 가격 대비 만족도에는 긍정적인 반응을 보였다.
특히 공기청정기와 무선청소기 상품에서는 흡입력, 소음 수준, 디자인에 대한 칭찬이 많았다.
배송 속도도 전반적으로 빠르다는 의견이 많았고, 지정일 배송을 편리하게 느낀 고객도 있었다.
다만 포장 상태에 대한 불만은 여러 상품군에서 반복적으로 나타났다.
일부 고객은 외부 박스가 찌그러져 도착했거나 완충재가 부족해 제품 파손을 걱정했다고 작성했다.
교환과 환불 절차에 대해서는 안내가 복잡하고 처리 상태를 확인하기 어렵다는 의견이 많았다.
고객센터 응답 속도도 주요 불만 요인으로 언급되었으며, 특히 주말 문의의 답변 지연이 자주 등장했다.
반면 상담원이 문제를 빠르게 해결해준 사례에서는 브랜드 신뢰도가 높아졌다는 평가도 있었다.
리뷰 작성자들은 제품 자체에는 만족하지만 구매 이후 문제가 생겼을 때의 지원 경험이 아쉽다고 정리했다.
재구매 의사가 있는 고객들도 포장 개선, 교환 절차 단순화, 배송 상태 알림 강화를 요구했다.
종합하면 상품 경쟁력은 유지되고 있으나, 물류와 사후 지원 경험을 개선하면 고객 만족도를 더 높일 수 있다.
""".strip()  # 요약과 키워드 추출에 사용할 긴 예시 텍스트입니다.

summary_prompt = ChatPromptTemplate.from_messages([  # 역할별 메시지로 프롬프트를 구성합니다.
    ("system", "너는 텍스트마이닝 분석가야."),  # 모델이 어떤 관점으로 답할지 정합니다.
    ("user", "다음 텍스트를 3문장 이내로 요약해줘.\n{text}")  # 실제 분석 요청과 입력 텍스트 자리입니다.
])

summary_chain = summary_prompt | llm | StrOutputParser()  # 프롬프트 -> 모델 -> 문자열 출력 순서로 체인을 만듭니다.
summary = summary_chain.invoke({"text": sample_text})  # {text} 자리에 sample_text를 넣어 체인을 실행합니다.
print(summary)  # 요약 결과를 출력합니다.


### 실무 예시: 오늘 경제 뉴스 요약

`naver_economy_news.csv`에 저장된 뉴스 제목과 본문을 LLM에 전달해 오늘 경제 뉴스의 흐름을 요약해 봅니다.  
기사 전체 본문을 모두 넣으면 입력이 길어질 수 있으므로, 아래 예시는 기사 수와 본문 길이를 제한해 실습 비용과 실행 시간을 조절합니다.


In [ ]:
news_df = pd.read_csv("naver_economy_news.csv")  # 오늘 수집한 네이버 경제 뉴스 CSV를 읽습니다.
news_df = news_df.dropna(subset=["title"]).copy()  # 제목이 없는 행은 요약 대상에서 제외합니다.
news_df["body"] = news_df["body"].fillna("")  # 본문이 비어 있는 경우 빈 문자열로 처리합니다.

article_limit = 12  # 요약에 사용할 기사 수입니다. 전체를 쓰고 싶으면 값을 늘려 보세요.
body_char_limit = 700  # 기사 1건당 본문에서 사용할 글자 수입니다.

news_sample = news_df.head(article_limit).copy()  # 입력 길이를 조절하기 위해 일부 기사만 사용합니다.

article_blocks = []  # LLM에 전달할 기사 묶음을 저장합니다.
for idx, row in news_sample.iterrows():
    article_blocks.append(
        f"[{idx + 1}] 제목: {row['title']}\n"
        f"언론사: {row.get('press', '')}\n"
        f"기자: {row.get('journalist', '')}\n"
        f"시간: {row.get('time', '')}\n"
        f"본문 일부: {row['body'][:body_char_limit]}"
    )

news_text = "\n\n".join(article_blocks)  # 여러 기사 내용을 하나의 입력 텍스트로 합칩니다.

news_summary_prompt = ChatPromptTemplate.from_messages([  # 오늘 뉴스 요약용 프롬프트를 구성합니다.
    ("system", "너는 한국 경제 뉴스를 정리하는 데이터 저널리스트야."),
    ("user", """
다음은 오늘 수집한 경제 뉴스 제목과 본문 일부야.
중복되는 내용은 묶고, 오늘 뉴스의 핵심 흐름을 한국어로 요약해줘.

출력 형식:
1. 한줄 요약: 오늘 경제 뉴스의 가장 큰 흐름 1문장
2. 핵심 이슈 3가지: 각 이슈를 bullet로 정리
3. 주목할 기업/기관: 기사에서 많이 언급되거나 의미 있는 기업·기관
4. 수업 토론 질문: 뉴스 데이터를 더 분석하기 위한 질문 2개

뉴스 데이터:
{news_text}
""")
])

news_summary_chain = news_summary_prompt | llm | StrOutputParser()  # 프롬프트 -> 모델 -> 문자열 출력 체인입니다.
today_news_summary = news_summary_chain.invoke({"news_text": news_text})  # 오늘 뉴스 요약을 생성합니다.

print(today_news_summary)  # 생성된 오늘 뉴스 요약을 출력합니다.


### 문제 1. 요약 관점 바꾸기

위 요약 프롬프트를 수정해 다음 형식으로 출력해 보세요.

- 핵심 긍정 의견
- 핵심 부정 의견
- 개선 제안

<details>
<summary>힌트 보기</summary>

`user` 메시지에 원하는 출력 항목을 직접 적으면 됩니다.

```python
("user", "긍정 의견, 부정 의견, 개선 제안으로 나눠 요약해줘.\n{text}")
```
</details>


## 3. 키워드 추출

키워드 추출은 문서에서 중요한 단어와 짧은 구를 뽑아 이슈를 요약하는 작업입니다.  
LLM을 사용할 때는 결과를 표로 다루기 쉽도록 JSON 형식으로 받는 것이 편합니다.


In [ ]:
keyword_parser = JsonOutputParser()  # 모델 응답을 파이썬 dict/list로 바꿔주는 JSON 파서입니다.

keyword_prompt = ChatPromptTemplate.from_messages([  # 키워드 추출용 프롬프트를 구성합니다.
    ("system", "너는 한국어 텍스트마이닝 분석가야."),  # 모델의 역할을 지정합니다.
    ("user", "키워드 5개를 JSON으로 추출해줘. 형식: {{\"keywords\": [{{\"keyword\": \"...\", \"reason\": \"...\"}}]}}\n{text}")  # 출력 형식과 분석할 텍스트를 전달합니다.
])

keyword_chain = keyword_prompt | llm | keyword_parser  # 프롬프트 -> 모델 -> JSON 파서 순서로 연결합니다.
keyword_result = keyword_chain.invoke({"text": sample_text})  # 예시 텍스트에서 키워드를 추출합니다.
keyword_result  # 추출된 JSON 결과를 확인합니다.


In [ ]:
keyword_df = pd.DataFrame(keyword_result["keywords"])  # keywords 리스트를 표 형태로 변환합니다.
keyword_df  # 키워드와 추출 이유를 데이터프레임으로 확인합니다.


## 4. 감정 분류

감정 분류는 리뷰나 댓글을 긍정, 부정, 중립으로 나누는 작업입니다.  
LLM은 문장의 표현을 읽고 분류할 수 있지만, 결과 기준을 명확히 적어야 일관성이 좋아집니다.


In [ ]:
review_df = pd.DataFrame({  # 감정 분류에 사용할 짧은 리뷰 예시를 만듭니다.
    "text": [
        "배송이 정말 빨라서 만족합니다.",
        "제품은 괜찮지만 포장이 너무 허술했어요.",
        "교환 신청을 했는데 답변이 너무 늦습니다.",
        "가격 대비 품질이 좋아서 재구매하고 싶어요.",
        "아직 사용 전이라 평가는 어렵습니다.",
    ]
})
review_df  # 분류 대상 리뷰 목록을 확인합니다.


In [ ]:
sentiment_prompt = ChatPromptTemplate.from_messages([  # 감정 분류용 프롬프트를 구성합니다.
    ("system", "너는 고객 리뷰를 분석하는 분류기야."),  # 모델의 역할을 분류기로 고정합니다.
    ("user", "감정을 positive, negative, neutral 중 하나로 JSON 분류해줘. 형식: {{\"sentiment\": \"...\", \"reason\": \"...\"}}\n리뷰: {text}")  # 리뷰 1건과 원하는 JSON 형식을 전달합니다.
])

sentiment_chain = sentiment_prompt | llm | JsonOutputParser()  # 프롬프트, 모델, JSON 파서를 하나의 체인으로 연결합니다.

sentiment_rows = []  # 리뷰별 분류 결과를 저장할 리스트입니다.
for text in review_df["text"]:  # 리뷰를 한 건씩 꺼내 모델에 전달합니다.
    sentiment_rows.append(sentiment_chain.invoke({"text": text}))  # 각 리뷰의 감정과 근거를 저장합니다.

sentiment_df = pd.concat([review_df, pd.DataFrame(sentiment_rows)], axis=1)  # 원본 리뷰와 분류 결과를 옆으로 붙입니다.
sentiment_df  # 최종 감정 분류 결과를 확인합니다.


## 5. 카테고리 분류

카테고리 분류는 상담 문의, 뉴스 기사, VOC를 미리 정한 업무 분류로 나누는 작업입니다.  
프롬프트에 가능한 카테고리 목록을 넣으면, 분류 기준을 더 안정적으로 만들 수 있습니다.


In [ ]:
inquiry_df = pd.DataFrame({  # 카테고리 분류에 사용할 고객 문의 예시를 만듭니다.
    "text": [
        "주문한 상품이 아직 도착하지 않았습니다.",
        "불량 제품을 받아서 환불하고 싶습니다.",
        "쿠폰 적용이 안 되는데 확인 부탁드립니다.",
        "제품 사용 중 전원이 자꾸 꺼집니다.",
        "회원 탈퇴 메뉴가 어디 있는지 모르겠습니다.",
    ]
})

categories = ["배송", "환불/교환", "가격/쿠폰", "상품품질", "계정/기타"]  # 모델이 선택할 수 있는 카테고리 목록입니다.
inquiry_df  # 분류 대상 문의 목록을 확인합니다.


In [ ]:
category_prompt = ChatPromptTemplate.from_messages([  # 카테고리 분류용 프롬프트를 구성합니다.
    ("system", "너는 고객 문의를 업무 카테고리로 분류하는 분석가야."),  # 모델의 역할을 업무 분류 분석가로 지정합니다.
    ("user", "카테고리 중 하나로 JSON 분류해줘. 형식: {{\"category\": \"...\", \"reason\": \"...\"}}\n카테고리: {categories}\n문의: {text}")  # 선택 가능한 카테고리와 문의 1건을 전달합니다.
])

category_chain = category_prompt | llm | JsonOutputParser()  # 프롬프트 -> 모델 -> JSON 파서 체인을 만듭니다.

category_rows = []  # 문의별 분류 결과를 저장할 리스트입니다.
for text in inquiry_df["text"]:  # 문의를 한 건씩 분류합니다.
    category_rows.append(category_chain.invoke({
        "categories": ", ".join(categories),  # 카테고리 리스트를 쉼표로 이어진 문자열로 전달합니다.
        "text": text,  # 현재 분류할 문의 문장입니다.
    }))

category_df = pd.concat([inquiry_df, pd.DataFrame(category_rows)], axis=1)  # 원본 문의와 분류 결과를 합칩니다.
category_df  # 최종 카테고리 분류 결과를 확인합니다.


## 6. OpenAI 임베딩으로 문서 유사도 계산

9번 파일에서는 텍스트를 벡터로 바꾸고 벡터 간 유사도를 계산하는 개념을 배웠습니다.  
이번에는 OpenAI 임베딩 모델로 문장을 벡터로 바꾼 뒤, 코사인 유사도를 직접 계산합니다.

이 방식은 검색, 추천, 중복 문서 탐지, RAG의 검색 단계에서 핵심적으로 사용됩니다.


In [ ]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")  # 문장을 임베딩 벡터로 바꿀 모델입니다.

text_docs = [  # 검색 대상이 되는 문서 목록입니다.
    "배송이 지연되어 고객 불만이 증가하고 있다.",
    "포장재가 약해 상품 파손 사례가 보고되었다.",
    "가격 할인 이벤트 이후 신규 고객 유입이 늘었다.",
    "환불 절차가 복잡해서 고객센터 문의가 많아졌다.",
    "제품 품질 만족도가 높아 재구매 의사가 증가했다.",
]

query = "교환과 환불 과정이 불편하다는 고객 의견"  # 검색 질문 또는 사용자의 관심 문장입니다.

doc_vectors = embedding_model.embed_documents(text_docs)  # 문서 목록을 각각 임베딩 벡터로 변환합니다.
query_vector = embedding_model.embed_query(query)  # 검색 문장도 같은 임베딩 공간의 벡터로 변환합니다.

print("문서 개수:", len(doc_vectors))  # 생성된 문서 벡터 개수를 확인합니다.
print("임베딩 차원:", len(query_vector))  # 임베딩 벡터의 길이를 확인합니다.
print("쿼리 벡터 앞 5개 값:", query_vector[:5])  # 벡터가 실제 숫자 배열인지 일부 값을 확인합니다.


In [ ]:
def cosine_similarity(vec_a, vec_b):  # 두 벡터의 코사인 유사도를 계산하는 함수입니다.
    a = np.array(vec_a)  # 첫 번째 벡터를 numpy 배열로 바꿉니다.
    b = np.array(vec_b)  # 두 번째 벡터를 numpy 배열로 바꿉니다.
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))  # 내적을 벡터 크기로 나눠 유사도를 계산합니다.

similarities = [cosine_similarity(query_vector, doc_vector) for doc_vector in doc_vectors]  # 쿼리와 각 문서의 유사도를 계산합니다.

similarity_df = pd.DataFrame({  # 문서별 유사도 결과를 표로 정리합니다.
    "query": query,
    "document": text_docs,
    "similarity": similarities,
}).sort_values("similarity", ascending=False)  # 유사도가 높은 문서부터 정렬합니다.

similarity_df  # 검색 결과처럼 가장 관련 높은 문서를 확인합니다.


### 문제 2. 나만의 임베딩 검색 만들기

아래 중 하나를 바꿔 실행해 보세요.

- `query`를 `배송 문제를 해결하고 싶다`로 변경
- `text_docs`에 새로운 문장 2개 추가
- 유사도가 가장 높은 문서 3개만 출력

<details>
<summary>힌트 보기</summary>

```python
similarity_df.head(3)
```
</details>


## 체크포인트

- LLM은 요약, 키워드 추출, 감정 분류, 카테고리 분류처럼 사람이 기준을 읽고 판단하던 작업을 빠르게 자동화할 수 있습니다.
- 출력 형식을 JSON으로 고정하면 결과를 데이터프레임으로 정리하기 쉽습니다.
- OpenAI 임베딩은 문장을 의미 벡터로 바꾸며, 코사인 유사도로 문서 검색과 추천을 구성할 수 있습니다.
- LLM 결과는 항상 검토가 필요합니다. 중요한 업무에서는 샘플 검수, 기준 문서화, 재현성 확인을 함께 해야 합니다.
